**Project: Movie Recommendation System**

Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast   #Converts stringified lists/dictionaries into Python lists/dicts
from sklearn.feature_extraction.text import CountVectorizer  #to convert text into vectors
from sklearn.metrics.pairwise import cosine_similarity   #to find how similar two movies are
import pickle

Step 2: Load Data

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')


In [ ]:
# Merge both datasets on the title column
movies = movies.merge(credits, on='title')
movies.head()
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

Step 3: Select Important Features

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']] # keep only the required columns
movies.dropna(inplace=True)

/tmp/ipython-input-30-1350021100.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies.dropna(inplace=True)


Step 4: Data Cleaning Functions

In [ ]:
#These functions extract useful text from complex JSON-like columns:
def convert(text):   #Har JSON object ka 'name' uthao.
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):  #Sirf top 3 actors lo.
    return [i['name'] for i in ast.literal_eval(text)][:3]  # top 3 actors

def fetch_director(text):   #Crew mein se director nikaalo.
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

def remove_spaces(L):   #Tom Hanks ko TomHanks bana do
    return [i.replace(" ", "") for i in L]

Step 5: Apply Cleaning

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)

movies['genres'] = movies['genres'].apply(remove_spaces)
movies['keywords'] = movies['keywords'].apply(remove_spaces)
movies['cast'] = movies['cast'].apply(remove_spaces)
movies['crew'] = movies['crew'].apply(remove_spaces)

Step 6: Create 'tags' column for NLP

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Drop old columns
new_df = movies[['movie_id', 'title', 'tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

/tmp/ipython-input-33-38903832.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


Step 7: Text Vectorization

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()


Step 8: Similarity Matrix

In [ ]:
similarity = cosine_similarity(vectors)

Step 9: Recommendation Function

In [ ]:
def recommend(movie):
    movie = movie.lower()
    if movie not in new_df['title'].str.lower().values:
        print("Movie not found in database.")
        return

    idx = new_df[new_df['title'].str.lower() == movie].index[0]
    distances = list(enumerate(similarity[idx]))
    movies_list = sorted(distances, reverse=True, key=lambda x: x[1])[1:6]

    print(f"\nTop 5 movies similar to '{movie.title()}':\n")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

Step 10: Save Files for Deployment

In [ ]:
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))


Test the Recommender

In [ ]:
recommend('Avatar') #Try multiple movie titles e.g., 'Avatar', 'The Dark Knight Rises', 'Spectre', etc.



Top 5 movies similar to 'Avatar':

Titan A.E.
Independence Day
Small Soldiers
Ender's Game
Aliens vs Predator: Requiem
